In [ ]:
import torch
import torch.nn as nn
from torchsummary import summary

In [ ]:
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
device

In [ ]:
x = torch.rand((4,1,128,128,128)) # batch, channel, depth, height, width
conv3d_block = nn.Sequential(
    nn.Conv3d(x.shape[1],16,3,1,padding = 1)
    nn.BatchNorm3d(16),
    nn.ReLU(),
)

summary(conv3d_block, x.shape[1:], device = 'gpu')
conv3d_block

In [ ]:
x = torch.rand((4,1,128,128,128))
conv3d_block = nn.Sequential(
    nn.ConvTranspose3d(x.shape[1],16,2,2),
    nn.BatchNorm3d(16),
    nn.ReLU(),
)
summary(conv3d_block, x.shape[1:], device = 'gpu')
conv3d_block

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size = 3, stride =1, padding =1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size, stride, padding),
            nn.BatchNorm3d(out_channels),
            nn.PReLU()
        )

    def forward(self, x):
        return self.net(x)

class BigBlock(nn.Module):
    def __init__(self, depth, in_channels, out_channels):
        super().__init__()
        self.layers = nn.ModuleList([])
        for _ in range(depth):
            self.layers.append(ConvBlock(in_channels, out_channels))
            in_channels = out_channels

    def forward(self, x):
        for l in self.layers:
            x = l(x)
        return x

class VNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = BigBlock(depth = 1, in_channels = 1, out_channels = 16)
        self.expand_ch1 = nn.Conv3d(16,32,kernel_size=3, stride=1, padding=1)
        self.down1 = nn.Conv3d(16,32,kernel_size=2, stride=2)

        self.enc2 = BigBlock(depth = 2, in_channels = 32, out_channels = 32)
        self.expand_ch2 = nn.Conv3d(32,64,kernel_size=3, stride=1, padding=1)
        self.down2 = nn.Conv3d(32,64,kernel_size=2, stride=2)

        self.enc3 = BigBlock(depth = 3, in_channels = 64, out_channels = 64)
        self.expand_ch3 = nn.Conv3d(64,128,kernel_size=3, stride=1, padding=1)
        self.down3 = nn.Conv3d(64,128,kernel_size=2, stride=2)

        self.enc4 = BigBlock(depth = 3, in_channels = 128, out_channels = 128)
        self.expand_ch4 = nn.Conv3d(128,256,kernel_size=3, stride=1, padding=1)
        self.down4 = nn.Conv3d(128,256,kernel_size=2, stride=2)

        self.enc5 = BigBlock(depth = 3, in_channels = 256, out_channels = 256)
        self.up5 = nn.ConvTranspose3d(256,256,kernel_size=2,stride=2)

        self.dec4 = BigBlock(depth =3, in_channels=256, out_channels = 256)
        self.up4 = nn.ConvTranspose3d(256,128,kernel_size=2,stride=2)

        self.dec3 = BigBlock(depth =3, in_channels=128, out_channels = 128)
        self.up3 = nn.ConvTranspose3d(128,64,kernel_size=2,stride=2)

        self.dec2 = BigBlock(depth =2, in_channels=64, out_channels = 64)
        self.up2 = nn.ConvTranspose3d(64,32,kernel_size=2,stride=2)

        self.dec1 = BigBlock(depth =1, in_channels=32, out_channels = 32)

        self.conv = nn.Conv3d(32,3,1,1)

    def forward(self,x):
        enc1_res = x
        enc1 = self.enc1(x)
        enc1 += enc1_res
        
        enc2_res = self.down1(enc1)
        enc2 = self.enc2(enc2_res)
        enc2 += enc2_res
        
        enc3_res = self.down2(enc2)
        enc3 = self.enc3(enc3_res)
        enc3 += enc3_res

        enc4_res = self.down3(enc3)
        enc4 = self.enc4(enc4_res)
        enc4 += enc4_res
        
        enc5_res = self.down4(enc4)
        enc5 = self.enc5(enc5_res)
        enc5 += enc5_res
        
        dec4_res = self.up5(enc5)
        enc4 = self.expand_ch4(enc4)
        dec4 = dec4_res + enc4
        dec4 = self.dec4(dec4)
        dec4 += dec4_res
        
        dec3_res = self.up4(enc4)
        enc3 = self.expand_ch3(enc3)
        dec3 = dec3_res + enc3
        dec3 = self.dec3(dec3)
        dec3 += dec3_res
        
        dec2_res = self.up3(enc3)
        enc2 = self.expand_ch2(enc2)
        dec2 = dec2_res + enc2
        dec2 = self.dec2(dec2)
        dec2 += dec2_res
        
        dec1_res = self.up2(enc2)
        enc1 = self.expand_ch1(enc1)
        dec1 = dec1_res + enc1
        dec1 = self.dec1(dec1)
        dec1 += dec1_res
        
        outputs = self.conv(dec1)
        
        return outputs

model = VNet()
x = torch.rand((4,1,64,128,128))
summary(model, x.shape[1:], device='cpu')
model